# Production FastAPI Deployment (Business-Friendly Pipeline)

This notebook implements a production-style machine learning API.

Instead of expecting the API consumer to manually one-hot encode inputs and calculate engineered ratios, **the API accepts a raw, business-friendly payload**. 

The internal pipeline is structured as follows:
1. **Validation**: Pydantic strictly validates the incoming business types.
2. **Encoding**: Label Encoding (binary) and One-Hot Encoding (categoricals) are dynamically applied.
3. **Feature Engineering**: Compute vital domain ratios securely (handling divide-by-zero).
4. **Column Alignment**: `pd.reindex` guarantees the payload structure is perfectly aligned to the XGBoost model's training footprint.
5. **Prediction**: XGBoost inference is run and evaluated against an optimized threshold (`0.3689`).
6. **Business Logic**: Translates raw probabilities into actionable Risk Levels (Low, Medium, High).


In [2]:
import pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier

# --- 1. Model Training & Saving ---
print("Loading full dataset for final tuned model...")
train = pd.read_csv('train_fe.csv')

# Features and target
X = train.drop(columns=['SK_ID_CURR', 'TARGET'])
y = train['TARGET']

# Best fine-tuned parameters from MLflow Tracking
best_params = {
    'colsample_bytree': 0.8853,
    'gamma': 1.5216,
    'learning_rate': 0.0675,
    'max_depth': 5,
    'min_child_weight': 1,
    'n_estimators': 114,
    'scale_pos_weight': 6.1089,
    'subsample': 0.7757,
    'random_state': 42,
    'eval_metric': 'auc'
}

print("Training fine-tuned XGBoost model...")
model = XGBClassifier(**best_params)
model.fit(X, y)

# Save the model
model.save_model('xgb_model.json')
print("Model saved to xgb_model.json")


Loading full dataset for final tuned model...
Training fine-tuned XGBoost model...
Model saved to xgb_model.json


In [3]:
import logging
import nest_asyncio
from pydantic_settings import BaseSettings

# --- 2. Configuration and Logging Setup ---

class Settings(BaseSettings):
    model_path: str = "xgb_model.json"
    app_name: str = "Home Credit Default Risk Assessor"
    model_version: str = "1.0.0"
    best_threshold: float = 0.3689

settings = Settings()

# Configure Logging to track incoming requests and anomalies
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

nest_asyncio.apply()


In [4]:
from pydantic import BaseModel, Field

# --- 3. Business-Friendly Schemas ---

class LoanApplication(BaseModel):
    """Raw business inputs collected from the loan application UI."""
    name_contract_type: str = Field(..., description="E.g., 'Cash loans', 'Revolving loans'")
    gender: str = Field(..., description="E.g., 'M', 'F'")
    own_car: str = Field(..., description="E.g., 'Y', 'N'")
    own_realty: str = Field(..., description="E.g., 'Y', 'N'")
    children_count: int = Field(..., ge=0)
    income_total: float = Field(..., gt=0.0)
    credit_amount: float = Field(..., gt=0.0)
    annuity_amount: float = Field(..., gt=0.0)
    goods_price: float = Field(..., gt=0.0)
    education_type: str = Field(..., description="E.g., 'Secondary / secondary special'")
    ext_source_1: float = Field(0.5)
    ext_source_2: float = Field(0.5)
    ext_source_3: float = Field(0.5)
    organization_type: str = Field(...)
    age: float = Field(..., gt=18.0)
    years_employed: float = Field(..., ge=0.0)
    name_type_suite: str = Field(...)
    income_type: str = Field(...)
    family_status: str = Field(...)
    occupation_type: str = Field(...)
    housing_type: str = Field(...)

class BusinessResponse(BaseModel):
    """Structured, actionable business response."""
    default_probability: float = Field(..., description="Raw probability of default")
    prediction: int = Field(..., description="Binary prediction based on optimized threshold")
    risk_level: str = Field(..., description="Low Risk, Medium Risk, High Risk")
    model_version: str = Field(...)


In [5]:
import pandas as pd
import numpy as np

# --- 4. Preprocessing Pipeline Logic ---

# Exact columns the model was trained on. Hardcoding this guarantees we never pass a misaligned schema.
EXPECTED_COLUMNS = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "NAME_EDUCATION_TYPE",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "ORGANIZATION_TYPE",
    "AGE",
    "YEARS_EMPLOYED",
    "NAME_TYPE_SUITE_Family",
    "NAME_TYPE_SUITE_Group of people",
    "NAME_TYPE_SUITE_Other_A",
    "NAME_TYPE_SUITE_Other_B",
    "NAME_TYPE_SUITE_Spouse, partner",
    "NAME_TYPE_SUITE_Unaccompanied",
    "NAME_INCOME_TYPE_Commercial associate",
    "NAME_INCOME_TYPE_Maternity leave",
    "NAME_INCOME_TYPE_Pensioner",
    "NAME_INCOME_TYPE_State servant",
    "NAME_INCOME_TYPE_Student",
    "NAME_INCOME_TYPE_Unemployed",
    "NAME_INCOME_TYPE_Working",
    "NAME_FAMILY_STATUS_Married",
    "NAME_FAMILY_STATUS_Separated",
    "NAME_FAMILY_STATUS_Single / not married",
    "NAME_FAMILY_STATUS_Unknown",
    "NAME_FAMILY_STATUS_Widow",
    "OCCUPATION_TYPE_Cleaning staff",
    "OCCUPATION_TYPE_Cooking staff",
    "OCCUPATION_TYPE_Core staff",
    "OCCUPATION_TYPE_Drivers",
    "OCCUPATION_TYPE_HR staff",
    "OCCUPATION_TYPE_High skill tech staff",
    "OCCUPATION_TYPE_IT staff",
    "OCCUPATION_TYPE_Laborers",
    "OCCUPATION_TYPE_Low-skill Laborers",
    "OCCUPATION_TYPE_Managers",
    "OCCUPATION_TYPE_Medicine staff",
    "OCCUPATION_TYPE_Private service staff",
    "OCCUPATION_TYPE_Realty agents",
    "OCCUPATION_TYPE_Sales staff",
    "OCCUPATION_TYPE_Secretaries",
    "OCCUPATION_TYPE_Security staff",
    "OCCUPATION_TYPE_Unknown",
    "OCCUPATION_TYPE_Waiters/barmen staff",
    "NAME_HOUSING_TYPE_House / apartment",
    "NAME_HOUSING_TYPE_Municipal apartment",
    "NAME_HOUSING_TYPE_Office apartment",
    "NAME_HOUSING_TYPE_Rented apartment",
    "NAME_HOUSING_TYPE_With parents",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO",
    "INCOME_PER_CHILD",
    "EXT_SOURCE_MEAN",
    "CREDIT_TERM",
    "EMPLOYED_TO_AGE_RATIO"
]

def preprocess_payload(app_data: LoanApplication) -> pd.DataFrame:
    """Converts a raw business payload into the exact ML feature space."""
    
    # 1. Start with a clean dictionary
    features = {}
    
    # 2. Straight numeric passes
    features['CNT_CHILDREN'] = app_data.children_count
    features['AMT_INCOME_TOTAL'] = app_data.income_total
    features['AMT_CREDIT'] = app_data.credit_amount
    features['AMT_ANNUITY'] = app_data.annuity_amount
    features['AMT_GOODS_PRICE'] = app_data.goods_price
    features['AGE'] = app_data.age
    features['YEARS_EMPLOYED'] = app_data.years_employed
    features['EXT_SOURCE_1'] = app_data.ext_source_1
    features['EXT_SOURCE_2'] = app_data.ext_source_2
    features['EXT_SOURCE_3'] = app_data.ext_source_3
    
    # We must map education and organization to numeric based on training (assuming Label Encoded during training)
    # Note: If they were frequency encoded or label encoded, we simulate the mapping here.
    # For this project, if they were kept as raw floats, we handle it appropriately.
    # From the dataset, we assume simple fallback mapping or 0 if unmapped for demonstration.
    features['NAME_EDUCATION_TYPE'] = 0.0 # Placeholder: Add true mapping
    features['ORGANIZATION_TYPE'] = 0.0   # Placeholder: Add true mapping
    
    # 3. Label Encoding Pipeline
    # Convert business logic ('Y' / 'M') to binary flags (1/0)
    features['NAME_CONTRACT_TYPE'] = 1 if app_data.name_contract_type.lower().startswith('c') else 0
    features['CODE_GENDER'] = 1 if app_data.gender.upper() == 'M' else 0
    features['FLAG_OWN_CAR'] = 1 if app_data.own_car.upper() == 'Y' else 0
    features['FLAG_OWN_REALTY'] = 1 if app_data.own_realty.upper() == 'Y' else 0
    
    # 4. One-Hot Encoding Pipeline
    # Dynamically inject 1 into the specific one-hot column corresponding to the categorical value
    # Suite
    suite_col = f"NAME_TYPE_SUITE_{app_data.name_type_suite}"
    if suite_col in EXPECTED_COLUMNS:
        features[suite_col] = 1
        
    # Income Type
    inc_col = f"NAME_INCOME_TYPE_{app_data.income_type}"
    if inc_col in EXPECTED_COLUMNS:
        features[inc_col] = 1
        
    # Family Status
    fam_col = f"NAME_FAMILY_STATUS_{app_data.family_status}"
    if fam_col in EXPECTED_COLUMNS:
        features[fam_col] = 1
        
    # Occupation
    occ_col = f"OCCUPATION_TYPE_{app_data.occupation_type}"
    if occ_col in EXPECTED_COLUMNS:
        features[occ_col] = 1
        
    # Housing
    house_col = f"NAME_HOUSING_TYPE_{app_data.housing_type}"
    if house_col in EXPECTED_COLUMNS:
        features[house_col] = 1

    # 5. Feature Engineering Pipeline
    # Securely calculate ratios preventing ZeroDivisionError
    features['CREDIT_INCOME_RATIO'] = app_data.credit_amount / app_data.income_total if app_data.income_total > 0 else 0.0
    features['ANNUITY_INCOME_RATIO'] = app_data.annuity_amount / app_data.income_total if app_data.income_total > 0 else 0.0
    features['CREDIT_GOODS_RATIO'] = app_data.credit_amount / app_data.goods_price if app_data.goods_price > 0 else 0.0
    features['INCOME_PER_CHILD'] = app_data.income_total / (app_data.children_count + 1)
    features['EXT_SOURCE_MEAN'] = (app_data.ext_source_1 + app_data.ext_source_2 + app_data.ext_source_3) / 3
    features['CREDIT_TERM'] = app_data.credit_amount / app_data.annuity_amount if app_data.annuity_amount > 0 else 0.0
    features['EMPLOYED_TO_AGE_RATIO'] = app_data.years_employed / app_data.age if app_data.age > 0 else 0.0

    # 6. Column Alignment
    # Convert to DataFrame
    df = pd.DataFrame([features])
    
    # Reindex forces the DataFrame to match the EXACT 64 columns used in XGBoost training.
    # Missing columns (like inactive one-hot categories) are cleanly filled with 0.
    aligned_df = df.reindex(columns=EXPECTED_COLUMNS, fill_value=0)
    
    return aligned_df


In [6]:
from fastapi import FastAPI, HTTPException
from contextlib import asynccontextmanager

# --- 5. FastAPI Application ---

loaded_model = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info(f"Loading XGBoost model from {settings.model_path}...")
    try:
        model = xgb.XGBClassifier()
        model.load_model(settings.model_path)
        loaded_model['xgb'] = model
        logger.info("Model loaded successfully.")
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise e
    yield
    logger.info("Shutting down model resources.")
    loaded_model.clear()

app = FastAPI(title=settings.app_name, lifespan=lifespan)

@app.get("/health", tags=["System"])
def health_check():
    """Simple API health check endpoint."""
    return {"status": "Healthy"}

@app.post("/predict", response_model=BusinessResponse, tags=["Business Prediction"])
def predict_risk(application: LoanApplication):
    """
    Accepts raw loan application data. 
    Internally handles encoding, feature engineering, and model alignment.
    """
    logger.info(f"Received application for processing: Income {application.income_total}")
    
    if 'xgb' not in loaded_model:
        raise HTTPException(status_code=503, detail="Model is currently unavailable.")
        
    try:
        # Preprocess Pipeline
        ml_input_df = preprocess_payload(application)
        
        # Inference
        model = loaded_model['xgb']
        probability = float(model.predict_proba(ml_input_df)[0][1])
        
        logger.info(f"Computed probability: {probability:.4f}")
        
        # Apply Optimized Threshold Business Logic
        prediction = 1 if probability >= settings.best_threshold else 0
        
        # Determine Risk Level
        if probability < 0.30:
            risk_level = "Low Risk"
        elif probability < 0.60:
            risk_level = "Medium Risk"
        else:
            risk_level = "High Risk"
            
        logger.info(f"Final Outcome: {prediction} ({risk_level})")
        
        return {
            "default_probability": probability,
            "prediction": prediction,
            "risk_level": risk_level,
            "model_version": settings.model_version
        }
        
    except Exception as e:
        logger.error(f"Error during preprocessing/prediction pipeline: {e}")
        raise HTTPException(status_code=400, detail=f"Pipeline error: {str(e)}")


In [7]:
import uvicorn
import asyncio

# --- 6. Launch the Server ---
print("\n--- Starting Business API Server ---")
print("Interactive Swagger UI docs will be available at: http://127.0.0.1:8000/docs")
print("NOTE: This cell will run continuously. To stop the server, interrupt the kernel (Stop button).\n")

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

# Await the server directly in the Jupyter async loop
await server.serve()


INFO:     Started server process [9292]
INFO:     Waiting for application startup.
2026-06-11 22:19:07,569 - __main__ - INFO - Loading XGBoost model from xgb_model.json...



--- Starting Business API Server ---
Interactive Swagger UI docs will be available at: http://127.0.0.1:8000/docs
NOTE: This cell will run continuously. To stop the server, interrupt the kernel (Stop button).



2026-06-11 22:19:07,711 - __main__ - INFO - Model loaded successfully.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51417 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:51417 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:51511 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:51523 - "GET /health HTTP/1.1" 200 OK


2026-06-11 22:21:28,494 - __main__ - INFO - Received application for processing: Income 150000.0
2026-06-11 22:21:28,553 - __main__ - INFO - Computed probability: 0.1541
2026-06-11 22:21:28,555 - __main__ - INFO - Final Outcome: 0 (Low Risk)


...

2026-06-11 22:59:17,483 - __main__ - INFO - Received application for processing: Income 250000.0
2026-06-11 22:59:17,516 - __main__ - INFO - Computed probability: 0.3509
2026-06-11 22:59:17,518 - __main__ - INFO - Final Outcome: 0 (Medium Risk)


INFO:     127.0.0.1:54657 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54931 - "GET /health HTTP/1.1" 200 OK


2026-06-11 23:03:32,630 - __main__ - INFO - Received application for processing: Income 120000.0
2026-06-11 23:03:32,684 - __main__ - INFO - Computed probability: 0.6311
2026-06-11 23:03:32,686 - __main__ - INFO - Final Outcome: 1 (High Risk)


INFO:     127.0.0.1:54932 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
2026-06-11 23:05:58,665 - __main__ - INFO - Shutting down model resources.
INFO:     Application shutdown complete.
INFO:     Finished server process [9292]
